# Gradient Textural MEB — block_0

Calcule et visualise le **gradient textural** : là où la texture change dans une image, à partir des features `block_0` de TextureSAM.

Un gradient élevé = frontière texturale.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os, zipfile, tempfile
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
import torch
import cv2
from scipy.ndimage import gaussian_filter

ROOT       = Path('..').resolve()
SAM2_DIR   = ROOT / 'TextureSAM' / 'sam2'
sys.path.insert(0, str(SAM2_DIR))
os.chdir(ROOT)

from hydra.core.global_hydra import GlobalHydra
from hydra import initialize_config_dir
GlobalHydra.instance().clear()
initialize_config_dir(
    config_dir=str(SAM2_DIR / 'sam2' / 'configs'),
    version_base='1.2'
)

from sam2.build_sam import build_sam2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

CFG_PATH   = ROOT / 'PatchTagger_Output' / 'config' / 'config.json'
DB_PATH    = ROOT / 'data' / 'feature_database' / 'database_meb.h5'
IMG_DIR    = ROOT / 'Image_Ouassim'
CHECKPOINT = 'checkpoints/sam2.1_hiera_small_1.pt'
KEY        = 'block_0'

# 3 images aux textures les plus variées (7, 7 et 5 catégories distinctes)
TARGET_IMGS = [
    '310120-pat18-WholeMount-24.tif',
    '070525-JPB-MEB-EIHNValves-Ech6-ZigZag0100.tif',
    '060722-Nabila-JP-Valves-WholeMount-SAureus-pat04-1-22.tif',
]

with open(CFG_PATH) as f:
    cfg = json.load(f)
CATEGORIES = {int(k): v['name']  for k, v in cfg['available_categories'].items()}
CAT_COLORS = {int(k): v['color'] for k, v in cfg['available_categories'].items()}

print(f'IMG_DIR    : {IMG_DIR}')
print(f'CHECKPOINT : {CHECKPOINT}')
print(f'Images     : {TARGET_IMGS}')

## Chargement des pseudo-labels (HDF5)

In [ ]:
import h5py

# Charge toutes les positions et catégories depuis la base HDF5
with h5py.File(DB_PATH, 'r') as _h5:
    _db_imgs = _h5['metadata/image_names'][:]    # (N,) bytes
    _db_cats = _h5['metadata/category_ids'][:]   # (N,) int
    _db_pos  = _h5['metadata/positions'][:]      # (N, 4) : x_min, y_min, x_max, y_max


def get_patches(img_name):
    """Retourne (positions, cat_ids) pour une image donnée."""
    mask = _db_imgs == img_name.encode()
    return _db_pos[mask], _db_cats[mask]


def hex_to_rgba(hex_color, alpha=0.45):
    h = hex_color.lstrip('#')
    r, g, b = (int(h[i:i+2], 16) / 255 for i in (0, 2, 4))
    return (r, g, b, alpha)


print('DB HDF5 chargée —', len(_db_imgs), 'patches')
for img in TARGET_IMGS:
    pos, cats = get_patches(img)
    cat_names = sorted({CATEGORIES[c] for c in cats})
    print(f'  {img[:45]}  → {len(pos)} patches  |  {cat_names}')

## Chargement du modèle

In [ ]:
# ── Chargement checkpoint (fichier .pt ou répertoire archive PyTorch) ──────────
MODEL_CFG = 'sam2.1/sam2.1_hiera_s.yaml'


def _load_ckpt_state_dict(ckpt_path):
    p = Path(ckpt_path)
    if p.is_file():
        sd = torch.load(p, map_location='cpu', weights_only=False)
        return sd.get('model', sd)

    archive_dir = p / 'archive' if (p / 'archive').is_dir() else p
    with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as tmp:
        tmp_path = tmp.name
    with zipfile.ZipFile(tmp_path, 'w', compression=zipfile.ZIP_STORED) as zf:
        for fp in archive_dir.rglob('*'):
            if fp.is_file():
                info = zipfile.ZipInfo(str(fp.relative_to(archive_dir.parent)))
                info.date_time = (1980, 1, 1, 0, 0, 0)
                with open(fp, 'rb') as fh:
                    zf.writestr(info, fh.read())
    sd = torch.load(tmp_path, map_location='cpu', weights_only=False)
    os.unlink(tmp_path)
    return sd.get('model', sd)


def load_model(checkpoint, cfg=MODEL_CFG):
    base_ck = str(ROOT / 'checkpoints' / 'sam2.1_hiera_small')
    sam2 = build_sam2(cfg, ckpt_path=None, device=device, apply_postprocessing=False)
    sam2.load_state_dict(_load_ckpt_state_dict(base_ck), strict=False)
    ck_full = str(ROOT / checkpoint) if not Path(checkpoint).is_absolute() else checkpoint
    if Path(ck_full).resolve() != Path(base_ck).resolve():
        missing, unexpected = sam2.load_state_dict(
            _load_ckpt_state_dict(ck_full), strict=False
        )
        print(f'Fine-tuned weights  missing={len(missing)} unexpected={len(unexpected)}')
    sam2.eval()
    return sam2


model = load_model(CHECKPOINT)
print(f'Modèle : {CHECKPOINT}  '
      f'({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)')

## Hook block_0

In [ ]:
# ── Hook sur le premier bloc trunk — sortie (B, H, W, C) ──────────────────────
_grad_cap = {}
_grad_hook = model.image_encoder.trunk.blocks[0].register_forward_hook(
    lambda m, i, o: _grad_cap.update({KEY: o.detach()})
)
print(f'Hook posé sur image_encoder.trunk.blocks[0]  (key={KEY!r})')

## Prétraitement image

In [ ]:
# ── Prétraitement — ImageNet normalisation, resize 1024×1024 ──────────────────
_grad_IMG_SIZE = 1024
_grad_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_grad_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def _grad_preprocess(img_pil):
    """PIL Image → (1, 3, 1024, 1024) tensor sur device."""
    img = img_pil.convert('RGB').resize(
        (_grad_IMG_SIZE, _grad_IMG_SIZE), Image.BILINEAR
    )
    x = torch.from_numpy(np.array(img)).float() / 255.0
    x = x.permute(2, 0, 1)
    x = (x - _grad_MEAN) / _grad_STD
    return x.unsqueeze(0).to(device)

## Fonction gradient textural

In [ ]:
def texture_gradient(feat_map, mode='cosine', neighborhood=8):
    """
    feat_map : (H, W, C) feature map block_0
    Retourne : (H, W) carte de gradient textural

    Pour chaque position, distance moyenne aux voisins.
    """
    H, W, C = feat_map.shape

    # L2-normaliser chaque vecteur
    norms = np.linalg.norm(feat_map, axis=2, keepdims=True)
    feat_norm = feat_map / np.where(norms < 1e-8, 1.0, norms)

    _grad_result = np.zeros((H, W))

    if neighborhood == 4:
        offsets = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    else:  # 8-connexité
        offsets = [
            (-1, -1), (-1, 0), (-1, 1),
            ( 0, -1),          ( 0, 1),
            ( 1, -1), ( 1, 0), ( 1, 1),
        ]

    for di, dj in offsets:
        shifted = np.roll(np.roll(feat_norm, di, axis=0), dj, axis=1)

        if mode == 'cosine':
            sim = (feat_norm * shifted).sum(axis=2)   # (H, W)
            dist = 1.0 - sim
        else:  # euclidienne
            dist = np.linalg.norm(feat_norm - shifted, axis=2)

        _grad_result += dist

    _grad_result /= len(offsets)

    # Bords invalides via roll → mise à 0
    _grad_result[0, :]  = 0
    _grad_result[-1, :] = 0
    _grad_result[:, 0]  = 0
    _grad_result[:, -1] = 0

    return _grad_result

## Traitement par image

In [ ]:
for _grad_target_img in TARGET_IMGS:

    _grad_img_path = IMG_DIR / _grad_target_img
    if not _grad_img_path.exists():
        print(f'[SKIP] {_grad_target_img} — introuvable')
        continue

    # ── Forward pass ──────────────────────────────────────────────────────────
    _grad_img = Image.open(_grad_img_path).convert('RGB')
    _grad_orig_H, _grad_orig_W = _grad_img.height, _grad_img.width
    _grad_tensor = _grad_preprocess(_grad_img)

    _grad_cap.clear()
    with torch.no_grad():
        model.image_encoder(_grad_tensor)

    _grad_feat_map = _grad_cap[KEY][0].cpu().numpy()   # (H_feat, W_feat, C)
    print(f'{_grad_target_img}  feat_map={_grad_feat_map.shape}')

    # ── Gradient textural ─────────────────────────────────────────────────────
    _grad_map    = texture_gradient(_grad_feat_map, mode='cosine', neighborhood=8)
    _grad_smooth = gaussian_filter(_grad_map, sigma=1.0)
    _grad_norm   = (
        (_grad_smooth - _grad_smooth.min())
        / (_grad_smooth.max() - _grad_smooth.min() + 1e-8)
    )
    _grad_full   = cv2.resize(
        _grad_norm, (_grad_orig_W, _grad_orig_H), interpolation=cv2.INTER_LINEAR
    )
    _grad_img_gray = np.array(_grad_img.convert('L'))

    # ── Figure 1×3 : original / gradient / overlay ────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    axes[0].imshow(_grad_img_gray, cmap='gray')
    axes[0].set_title('Image originale', fontsize=11)
    axes[0].axis('off')
    im1 = axes[1].imshow(_grad_full, cmap='hot')
    axes[1].set_title('Gradient textural\nchaud = frontière texturale', fontsize=11)
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    axes[2].imshow(_grad_img_gray, cmap='gray')
    axes[2].imshow(_grad_full, cmap='hot', alpha=0.5)
    axes[2].set_title('Overlay\nfrontières sur image', fontsize=11)
    axes[2].axis('off')
    plt.suptitle(f'Gradient textural — {KEY} — {_grad_target_img}', fontsize=13)
    plt.tight_layout()
    plt.savefig(ROOT / 'outputs' / f'texture_gradient_{_grad_target_img[:20]}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # ── Figure — frontières seuillées (top 15% gradient) ─────────────────────
    _grad_threshold  = np.percentile(_grad_full, 85)
    _grad_boundaries = _grad_full > _grad_threshold

    fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    ax1.imshow(_grad_img_gray, cmap='gray')
    ax1.set_title('Image originale', fontsize=11)
    ax1.axis('off')
    ax2.imshow(_grad_img_gray, cmap='gray')
    _grad_overlay = np.zeros((*_grad_boundaries.shape, 4))
    _grad_overlay[_grad_boundaries] = [0, 1, 1, 0.8]
    ax2.imshow(_grad_overlay)
    ax2.set_title('Frontières texturales détectées\n(top 15% gradient)', fontsize=11)
    ax2.axis('off')
    plt.suptitle(f'Frontières texturales — {_grad_target_img}', fontsize=12)
    plt.tight_layout()
    plt.savefig(ROOT / 'outputs' / f'texture_boundaries_{_grad_target_img[:20]}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # ── Statistiques gradient ─────────────────────────────────────────────────
    print(f'=== {_grad_target_img} ===')
    print(f'  Gradient moyen   : {_grad_smooth.mean():.4f}')
    print(f'  Gradient max     : {_grad_smooth.max():.4f}')
    print(f'  % zones frontière: {_grad_boundaries.mean()*100:.1f}%')
    print(f'  → zones homogènes: {(1-_grad_boundaries.mean())*100:.1f}%\n')

    # ── Figure — Pseudo-labels (patches annotés depuis HDF5) ──────────────────
    _grad_pl_pos, _grad_pl_cats = get_patches(_grad_target_img)

    if len(_grad_pl_pos) == 0:
        print(f'  [INFO] Aucun pseudo-label pour {_grad_target_img}\n')
        continue

    fig3, (ax_pl1, ax_pl2) = plt.subplots(1, 2, figsize=(16, 7))

    ax_pl1.imshow(_grad_img_gray, cmap='gray')
    ax_pl1.set_title('Image originale', fontsize=11)
    ax_pl1.axis('off')

    ax_pl2.imshow(_grad_img_gray, cmap='gray')
    _grad_present_cats = sorted(set(_grad_pl_cats.tolist()))
    for (x_min, y_min, x_max, y_max), cat_id in zip(_grad_pl_pos, _grad_pl_cats):
        w, h = x_max - x_min, y_max - y_min
        rect = mpatches.FancyBboxPatch(
            (x_min, y_min), w, h,
            boxstyle='square,pad=0',
            linewidth=1.2,
            edgecolor=CAT_COLORS[int(cat_id)],
            facecolor=hex_to_rgba(CAT_COLORS[int(cat_id)], alpha=0.45),
        )
        ax_pl2.add_patch(rect)

    _grad_legend_handles = [
        mpatches.Patch(
            facecolor=hex_to_rgba(CAT_COLORS[c], alpha=0.7),
            edgecolor=CAT_COLORS[c],
            label=f'{c} — {CATEGORIES[c]}',
        )
        for c in _grad_present_cats
    ]
    ax_pl2.legend(handles=_grad_legend_handles, loc='upper right',
                  fontsize=8, framealpha=0.85)
    ax_pl2.set_title(f'Pseudo-labels — {len(_grad_pl_pos)} patches annotés',
                     fontsize=11)
    ax_pl2.axis('off')
    plt.suptitle(f'Pseudo-labels — {_grad_target_img}', fontsize=12)
    plt.tight_layout()
    plt.savefig(ROOT / 'outputs' / f'pseudolabels_{_grad_target_img[:20]}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    print(f'  Catégories présentes :')
    for c in _grad_present_cats:
        print(f'    {c:2d}  {CATEGORIES[c]:<30}  {(_grad_pl_cats == c).sum()} patches')
    print()

## Comparaison : Gradient Textural vs Intensité (Sobel)

> **Objectif** : trancher si `block_0` détecte la **texture** ou juste l'**intensité**.
> - Gradient textural ≈ Sobel → block_0 suit l'intensité
> - Gradient textural ≠ Sobel → block_0 capture la vraie texture

In [ ]:
from scipy.ndimage import sobel as _cmp_sobel
from scipy.stats import pearsonr as _cmp_pearsonr

_cmp_TARGET_IMG = '310120-pat18-WholeMount-24.tif'
_cmp_IMG_PATH   = IMG_DIR / _cmp_TARGET_IMG

# ── Gradient textural block_0 ──────────────────────────────────────────────────
_cmp_img      = Image.open(_cmp_IMG_PATH).convert('RGB')
_cmp_orig_H, _cmp_orig_W = _cmp_img.height, _cmp_img.width
_cmp_tensor   = _grad_preprocess(_cmp_img)

_grad_cap.clear()
with torch.no_grad():
    model.image_encoder(_cmp_tensor)

_cmp_feat_map     = _grad_cap[KEY][0].cpu().numpy()   # (H_feat, W_feat, C)
_cmp_grad_texture = texture_gradient(_cmp_feat_map, mode='cosine', neighborhood=8)
_cmp_grad_texture = gaussian_filter(_cmp_grad_texture, sigma=1.0)

# ── Gradient d'intensité (Sobel) ───────────────────────────────────────────────
_cmp_img_gray    = np.array(_cmp_img.convert('L')).astype(np.float32)
_cmp_sobel_x     = _cmp_sobel(_cmp_img_gray, axis=0)
_cmp_sobel_y     = _cmp_sobel(_cmp_img_gray, axis=1)
_cmp_grad_intens = np.sqrt(_cmp_sobel_x**2 + _cmp_sobel_y**2)
_cmp_grad_intens = gaussian_filter(_cmp_grad_intens, sigma=2.0)

# ── Downsample à la résolution block_0 (256×256) pour comparaison équitable ───
_cmp_H_feat, _cmp_W_feat = _cmp_feat_map.shape[:2]
_cmp_tex_256 = cv2.resize(_cmp_grad_texture,
                           (_cmp_W_feat, _cmp_H_feat),
                           interpolation=cv2.INTER_LINEAR)
_cmp_int_256 = cv2.resize(_cmp_grad_intens,
                           (_cmp_W_feat, _cmp_H_feat),
                           interpolation=cv2.INTER_LINEAR)


def _cmp_norm01(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-8)


_cmp_tex_n = _cmp_norm01(_cmp_tex_256)
_cmp_int_n = _cmp_norm01(_cmp_int_256)

# ── Corrélation Pearson ────────────────────────────────────────────────────────
_cmp_corr, _cmp_pval = _cmp_pearsonr(_cmp_tex_n.flatten(), _cmp_int_n.flatten())

print(f'=== Corrélation gradient textural vs intensité ===')
print(f'  Pearson r = {_cmp_corr:.3f}  (p={_cmp_pval:.2e})')
print()
if abs(_cmp_corr) > 0.7:
    print('  → CORRÉLATION FORTE')
    print("  → block_0 suit largement l'intensité")
    print("  → l'hypothèse \"ombres\" est plausible")
elif abs(_cmp_corr) > 0.4:
    print('  → corrélation modérée')
    print("  → block_0 capture l'intensité ET autre chose")
else:
    print('  → CORRÉLATION FAIBLE')
    print("  → block_0 détecte la TEXTURE, pas l'intensité")
    print("  → les frontières sont texturales, pas des ombres")

# ── Cartes de différence ───────────────────────────────────────────────────────
_cmp_diff = _cmp_tex_n - _cmp_int_n
# diff > 0 : texture change SANS changement d'intensité → vraie frontière texturale
# diff < 0 : intensité change SANS texture → ombre pure ignorée par block_0

# ── Figure 2×3 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(20, 11))

_cmp_img_gray_u8 = np.array(_cmp_img.convert('L'))
_cmp_img_small   = cv2.resize(_cmp_img_gray_u8,
                               (_cmp_W_feat, _cmp_H_feat),
                               interpolation=cv2.INTER_LINEAR)

# Ligne 0 : image | gradient textural | gradient intensité
axes[0, 0].imshow(_cmp_img_small, cmap='gray')
axes[0, 0].set_title('Image originale', fontsize=11)
axes[0, 0].axis('off')

im01 = axes[0, 1].imshow(_cmp_tex_n, cmap='hot')
axes[0, 1].set_title('Gradient TEXTURAL (block_0)', fontsize=11)
axes[0, 1].axis('off')
plt.colorbar(im01, ax=axes[0, 1], fraction=0.046)

im02 = axes[0, 2].imshow(_cmp_int_n, cmap='hot')
axes[0, 2].set_title('Gradient INTENSITÉ (Sobel)', fontsize=11)
axes[0, 2].axis('off')
plt.colorbar(im02, ax=axes[0, 2], fraction=0.046)

# Ligne 1 : scatter | texture sans intensité | intensité sans texture
axes[1, 0].scatter(
    _cmp_int_n.flatten()[::20],
    _cmp_tex_n.flatten()[::20],
    s=2, alpha=0.3, color='#1B4F72',
)
axes[1, 0].set_xlabel('Gradient intensité', fontsize=10)
axes[1, 0].set_ylabel('Gradient textural', fontsize=10)
axes[1, 0].set_title(f'Corrélation  r = {_cmp_corr:.3f}', fontsize=11)
axes[1, 0].grid(True, alpha=0.3)

im11 = axes[1, 1].imshow(np.clip(_cmp_diff, 0, 1), cmap='viridis')
axes[1, 1].set_title("Texture change SANS intensité\n(vraie frontière texturale)",
                      fontsize=11)
axes[1, 1].axis('off')
plt.colorbar(im11, ax=axes[1, 1], fraction=0.046)

im12 = axes[1, 2].imshow(np.clip(-_cmp_diff, 0, 1), cmap='viridis')
axes[1, 2].set_title("Intensité change SANS texture\n(ombre pure)",
                      fontsize=11)
axes[1, 2].axis('off')
plt.colorbar(im12, ax=axes[1, 2], fraction=0.046)

plt.suptitle(
    f'Texture vs Intensité — {_cmp_TARGET_IMG}\nCorrélation Pearson = {_cmp_corr:.3f}',
    fontsize=13,
)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'texture_vs_intensity.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== LECTURE DES CARTES DE DIFFÉRENCE ===')
print('Carte "texture sans intensité" (viridis clair) :')
print("  → block_0 voit une frontière là où l'œil")
print("    ne voit pas de changement d'intensité")
print("  → preuve que block_0 capture la VRAIE texture")
print()
print("Carte \"intensité sans texture\" :")
print("  → changements d'intensité que block_0 IGNORE")
print("  → block_0 ne suit pas aveuglément la luminosité")

## Nettoyage

In [ ]:
_grad_hook.remove()
print('Hook retiré.')

## Interprétation

```
Gradient élevé (chaud) = la texture change rapidement
  → frontière entre deux textures différentes
Gradient faible (sombre) = texture homogène
  → zone d'une seule texture

Les frontières détectées peuvent guider
le placement des points pour TextureSAM.
```